In [1]:
!pip install dialogflow gensim annoy tqdm stop_words pymorphy2 python-telegram-bot==13.3

In [2]:
!pip install pandarallel

In [3]:
!pip install ipywidgets

In [4]:
!pip install scikit-learn

In [6]:
pip freeze

absl-py==1.4.0
aiohttp==3.9.1
aiosignal==1.3.1
alabaster==0.7.13
albumentations==1.3.1
altair==4.2.2
annoy==1.17.3
anyascii==0.3.2
anyio==3.7.1
appdirs==1.4.4
APScheduler==3.6.3
argon2-cffi==23.1.0
argon2-cffi-bindings==21.2.0
array-record==0.5.0
arviz==0.15.1
astropy==5.3.4
astunparse==1.6.3
async-timeout==4.0.3
atpublic==4.0
attrs==23.2.0
audioread==3.0.1
autograd==1.6.2
Babel==2.14.0
backcall==0.2.0
beautifulsoup4==4.11.2
bidict==0.22.1
bigframes==0.18.0
bleach==6.1.0
blinker==1.4
blis==0.7.11
blosc2==2.0.0
bokeh==3.3.2
bqplot==0.12.42
branca==0.7.0
build==1.0.3
CacheControl==0.13.1
cachetools==5.3.2
catalogue==2.0.10
certifi==2023.11.17
cffi==1.16.0
chardet==5.2.0
charset-normalizer==3.3.2
chex==0.1.7
click==8.1.7
click-plugins==1.1.1
cligj==0.7.2
cloudpickle==2.2.1
cmake==3.27.9
cmdstanpy==1.2.0
colorcet==3.0.1
colorlover==0.3.0
colour==0.1.5
community==1.0.0b1
confection==0.1.4
cons==0.4.6
contextlib2==21.6.0
contourpy==1.2.0
contractions==0.1.73
cryptography==41.0.7
cufflinks==0

RESTART KERNEL

In [8]:
from google.colab import drive
drive.mount('./drive')

Drive already mounted at ./drive; to attempt to forcibly remount, call drive.mount("./drive", force_remount=True).


In [9]:
import os
from telegram.ext  import Updater, CommandHandler, MessageHandler, Filters
import string
from pymorphy2 import MorphAnalyzer
from stop_words import get_stop_words
import annoy
from collections.abc import Mapping
from gensim.models import Word2Vec, FastText
import pickle
import numpy as np
from tqdm.notebook import tqdm
import pandas as pd
import pickle

tqdm.pandas()

from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

INFO: Pandarallel will run on 1 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [10]:
# Скачать файл Otvety.txt по ссылке  (1,7G)
# https://drive.google.com/file/d/1DQL9ybca4USImUDaxxHmkIZNmClKBtKG/view

In [11]:
import mmap
import re

def get_num_lines(file_path):
    fp = open(file_path, "r+")
    buf = mmap.mmap(fp.fileno(), 0)
    lines = 0
    while buf.readline():
        lines += 1
    return lines

In [13]:
get_num_lines("/content/drive/MyDrive/Colab Notebooks/telegram_bot/Otvety.txt")

7550926

In [14]:
# Преобразование файла вопросов-ответов в строчный вид
if not os.path.isfile('/content/drive/MyDrive/Colab Notebooks/telegram_bot/prepared_answers.txt'):
  question = None
  written = False

  with open("/content/drive/MyDrive/Colab Notebooks/telegram_bot/prepared_answers.txt", "w") as fout:
      with open("/content/drive/MyDrive/Colab Notebooks/telegram_bot/Otvety.txt", "r") as fin:
          for line in tqdm(fin):
              if line.startswith("---"):
                  written = False
                  continue
              if not written and question is not None:
                  fout.write(question.replace("\t", " ").strip() + "===" + line.replace("\t", " "))
                  written = True
                  question = None
                  continue
              if not written:
                  question = line.strip()
                  continue

In [15]:
# Препроцессинг текста
def preprocess_txt(line):
    spls = "".join(i for i in line.strip() if i not in exclude).split()
    spls = [morpher.parse(i.lower())[0].normal_form for i in spls]
    spls = [i for i in spls if i not in sw and i != ""]
    return spls

In [18]:
# Обработка текста

sentences = []
morpher = MorphAnalyzer()
sw = set(get_stop_words("ru"))
exclude = set(string.punctuation)
#exclude = {}
c = 0

file_path_from = '/content/drive/MyDrive/Colab Notebooks/telegram_bot/prepared_answers.txt'
file_path_to = '/content/drive/MyDrive/Colab Notebooks/telegram_bot/Otvety2.txt'

if not os.path.isfile(file_path_to):

    N = get_num_lines(file_path_from)
    with open(file_path_to, mode = 'w') as fileto:
        with open(file_path_from) as filefrom:
            for k in tqdm(range(N)):
                line = filefrom.readline()
                if line == '': break
                spls = preprocess_txt(line)
                sentences.append(spls)
                c += 1
                if c > 50_000:
                    break
                fileto.write(' '.join(spls)+'\n')
    filefrom.close()
    fileto.close()

In [19]:
# Загрузить результат

sentences = []

file_path_from = '/content/drive/MyDrive/Colab Notebooks/telegram_bot/Otvety2.txt'
if os.path.isfile(file_path_from):
    N = get_num_lines(file_path_from)
    with open(file_path_to, mode = 'r') as filefrom:
        for k in tqdm(range(N)):
            line = filefrom.readline()
            if line == '': break
            sentences.append(line.split())
    filefrom.close()

  0%|          | 0/50000 [00:00<?, ?it/s]

In [20]:
vec = []
_ = [vec.extend(x)  for x in sentences[:100]]
vec = list(set(vec))
vec.sort()
vec[20:30]

['1542',
 '1548',
 '1588',
 '16ть',
 '1890—1907liul',
 '1898',
 '18мкака',
 '1914',
 '1мми',
 '2']

In [22]:
sentences[:10]

[['вопрос', 'тдв', 'отдыхать', 'лично', 'советовать', 'завести'],
 ['парень',
  'относиться',
  'цветной',
  'линза',
  'девушка',
  'зелёный',
  'глаз',
  'голубой',
  'вобщий',
  'прикалывать',
  'тема'],
 ['делать',
  'найти',
  '2',
  'миллион',
  'рубль',
  'счастие',
  'свалиться',
  'хороший',
  'пойти',
  'милиция',
  'заявить',
  'находка',
  'деньга',
  'тероть',
  'самый',
  'интересный',
  'неприменный',
  'искать',
  'поверьте',
  'найти',
  'видеть',
  'подобный',
  'нарваться',
  'бабушка',
  'помочий',
  'внук',
  'покупка',
  'квартира',
  'бандит',
  'разговаривать',
  'иначе',
  'бабушка',
  'милиция',
  'выбор',
  'шанс',
  'подарок',
  'выше',
  'котрый',
  'никто',
  'спросить',
  'хороший',
  'отдать',
  'хотяб',
  '500',
  'благотворительность',
  'дабы',
  'спугнуть',
  'удача'],
 ['эбу',
  'двенашка',
  'называться',
  'итэлма',
  'эбу',
  'эбу',
  '—',
  'электронный',
  'блок',
  'управление',
  'двигатель',
  'автомобиль',
  'название',
  '—',
  'контроллер

In [23]:
# Обучим модель FastText

file_path_from = '/content/drive/MyDrive/Colab Notebooks/telegram_bot/ft_model'
if not os.path.isfile(file_path_from):

    sentences = [i for i in tqdm(sentences) if len(i) > 2]
    modelFT = FastText(sentences=sentences, vector_size=100, min_count=1, window=5)
    modelFT.save("/content/drive/MyDrive/Colab Notebooks/telegram_bot/ft_model")
    ft_index = annoy.AnnoyIndex(100 ,'angular')

In [24]:
# Загрузить модель

modelFT = FastText.load("/content/drive/MyDrive/Colab Notebooks/telegram_bot/ft_model")
ft_index = annoy.AnnoyIndex(100 ,'angular')

In [25]:
list(set(get_stop_words("ru")))[:20]

['самому',
 'будем',
 'тобою',
 'раньше',
 'вокруг',
 'пятнадцатый',
 'него',
 'ваш',
 'в',
 'вы',
 'мне',
 'двенадцатый',
 'были',
 'лет',
 'эту',
 'теперь',
 'две',
 'как',
 'слишком',
 'самом']

In [26]:
# Создаем Индексы для вопросов-ответов

file_path_from = '/content/drive/MyDrive/Colab Notebooks/telegram_bot/speaker.ann'
if not os.path.isfile(file_path_from):
  morpher = MorphAnalyzer()
  sw = set(get_stop_words("ru"))
  exclude = set(string.punctuation)
  modelFT = FastText.load("/content/drive/MyDrive/Colab Notebooks/telegram_bot/ft_model")
  ft_index = annoy.AnnoyIndex(100 ,'angular')

  index_map = {}
  counter = 0
  with open("/content/drive/MyDrive/Colab Notebooks/telegram_bot/Otvety2.txt", "r") as f:
      for line in tqdm(f):
          n_ft = 0
          spls = line.split("===")
          index_map[counter] = spls[0]
          question = preprocess_txt(spls[0])
          vector_ft = np.zeros(100)
          for word in question:
              if word in modelFT.wv:
                  vector_ft += modelFT.wv[word]
                  n_ft += 1
          if n_ft > 0:
              vector_ft = vector_ft / n_ft
          ft_index.add_item(counter, vector_ft)
          counter += 1

          if counter > 50_000:
              break

  ft_index.build(10)
  ft_index.save('/content/drive/MyDrive/Colab Notebooks/telegram_bot/speaker.ann')

  with open("/content/drive/MyDrive/Colab Notebooks/telegram_bot/index_speaker.pkl", "wb") as f:
      pickle.dump(index_map, f)

In [27]:
#  Загрузим индексы
ft_index = annoy.AnnoyIndex(100, 'angular')
ft_index.load('/content/drive/MyDrive/Colab Notebooks/telegram_bot/speaker.ann')
index_map = pd.read_pickle("/content/drive/MyDrive/Colab Notebooks/telegram_bot/index_speaker.pkl")

In [28]:
np.random.permutation(100)

array([39, 51, 38, 87, 26,  7, 15, 32, 97, 80, 37,  8, 83, 34, 74, 36, 93,
       22, 99, 29, 82, 90, 41,  4, 79, 72, 20, 81, 19, 65, 27, 45, 67, 57,
       24, 25, 31,  3, 35, 44, 69, 13, 55, 73, 10, 86, 62, 40, 46, 76, 42,
        6, 17, 23, 54,  5, 28, 33, 84,  1, 75, 94,  0, 12, 53, 52, 64, 56,
        2, 88, 92, 85,  9, 78, 18, 71, 14, 11, 58, 98, 91, 70, 60, 30, 68,
       48, 66, 61, 89, 77, 96, 95, 16, 59, 47, 49, 50, 63, 21, 43])

In [29]:
#  Пример получения индексов
a = ft_index.get_nns_by_vector(np.random.permutation(100), 5, include_distances=True)
a

([36813, 33820, 44633, 35883, 44686],
 [1.2576992511749268,
  1.2589623928070068,
  1.2721468210220337,
  1.2829909324645996,
  1.2864819765090942])

In [30]:
[index_map[x] for x in a[0]]

['военный академияpвоенный академия p ul li военный академия — сербский телесериал кадет белградский военный академииli li военный академия одесса li li военный академия «людовика» li li военный академия гс раковск li li военный академия генеральный штаб вооружённый сила российский федерация li li военный академия императорский флот япония li li военный академия императорский армия япония li li военный академия карлберг li li военный академия рвсна пётр великий li li военный академия рхбз инженерный войско li li военный академия республика беларусь li li военный академия сша li li военный академия сербия li li военный академия береговой охрана сша li li военный академия бронетанковый войско li li военный академия воздушнокосмический оборона маршал советский союз жуков li li военный академия войсковой противовоздушный оборона вооружённый сила российский федерация li li военный академия фрунзе li li военный академия материальнотехнический обеспечение хрулёва li li военный академия связь 

In [31]:
# Создадим модель продуктовых данных
# https://gbcdn.mrgcdn.ru/uploads/asset/5209459/attachment/1b2f5aa57ff77e7c2d2ee26ceb09eb9e.csv
shop_data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/telegram_bot/ProductsDataset.csv")
#shop_data = shop_data.iloc[:5000, :]

shop_data['text'] = shop_data['title'] + " " + shop_data["descrirption"]
shop_data['text'] = shop_data['text'].progress_apply(lambda x: preprocess_txt(str(x)))
shop_data.head()

  0%|          | 0/35548 [00:00<?, ?it/s]

,title,descrirption,product_id,category_id,subcategory_id,properties,image_links,text
0,Юбка детская ORBY,"Новая, не носили ни разу. В реале красивей чем...",58e3cfe6132ca50e053f5f82,22.0,2211,"{'detskie_razmer_rost': '81-86 (1,5 года)'}",http://cache3.youla.io/files/images/360_360/58...,"[юбка, детский, orby, новый, носить, реал, кра..."
1,Ботильоны,"Новые,привезены из Чехии ,указан размер 40,но ...",5667531b2b7f8d127d838c34,9.0,902,"{'zhenskaya_odezhda_tzvet': 'Зеленый', 'visota...",http://cache3.youla.io/files/images/360_360/5b...,"[ботильон, новыепривезти, чехия, указать, разм..."
2,Брюки,Размер 40-42. Брюки почти новые - не знаю как ...,59534826aaab284cba337e06,9.0,906,{'zhenskaya_odezhda_dzhinsy_bryuki_tip': 'Брюк...,http://cache3.youla.io/files/images/360_360/59...,"[брюки, размер, 4042, брюки, новый, знать, мер..."
3,Продам детские шапки,"Продам шапки,кажда 200р.Розовая и белая проданны.",57de544096ad842e26de8027,22.0,2217,"{'detskie_pol': 'Девочкам', 'detskaya_odezhda_...",http://cache3.youla.io/files/images/360_360/57...,"[продать, детский, шапка, продать, шапкикажда,..."
4,Блузка,"Темно-синяя, 42 размер,состояние отличное,как ...",5ad4d2626c86cb168d212022,9.0,907,"{'zhenskaya_odezhda_tzvet': 'Синий', 'zhenskay...",http://cache3.youla.io/files/images/360_360/5a...,"[блузка, темносиний, 42, размерсостояние, отли..."


In [32]:
# Подготовка для создания модели для определения домена данных

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = CountVectorizer(ngram_range=(1, 2))

In [33]:
idxs = set(np.random.randint(0, len(index_map), len(shop_data)))
# Вопрос-ответный домен
negative_texts = [" ".join(preprocess_txt(index_map[i])) for i in tqdm(idxs)]
# Продуктовый домен
positive_texts = [" ".join(val) for val in tqdm(shop_data['text'].values)]

  0%|          | 0/25449 [00:00<?, ?it/s]

  0%|          | 0/35548 [00:00<?, ?it/s]

In [34]:
# ВО = 0, Прод = 1

dataset = negative_texts + positive_texts
labels = np.zeros(len(dataset))
labels[len(negative_texts):] = np.ones(len(positive_texts))

In [35]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(dataset, labels, test_size=0.2,
                                                    stratify=labels, random_state=13)

In [36]:
# Модель

x_train_vec = vectorizer.fit_transform(X_train)
x_test_vec = vectorizer.transform(X_test)

lr = LogisticRegression().fit(x_train_vec, y_train)

In [37]:
# Качество

from sklearn.metrics import accuracy_score
accuracy_score(y_true=y_test, y_pred=lr.predict(x_test_vec))

0.9833606557377049

Добавим IDF взвешивание (для каждого слова найдем IDF вес)

In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vect = TfidfVectorizer().fit(X_train)

In [39]:
np.mean(tfidf_vect.idf_)

10.78052970587605

In [40]:
idfs = {v[0]: v[1] for v in zip(tfidf_vect.vocabulary_, tfidf_vect.idf_)}

In [41]:
len(idfs.keys())

177658

In [42]:
list(idfs.keys())[:5]

['сумка', 'кавалли', 'оригинал', 'новый', 'оригинальный']

In [43]:
list(idfs.values())[:5]

[8.799712333840839,
 6.967130870092529,
 10.18600669496073,
 11.102297426834886,
 11.102297426834886]

In [44]:
# Создаем Индексы для продуктовых данных

file_path_from = '/content/drive/MyDrive/Colab Notebooks/telegram_bot/shop.ann'
if not os.path.isfile(file_path_from):


    ft_index_shop = annoy.AnnoyIndex(100 ,'angular')

    midf = np.mean(tfidf_vect.idf_)

    index_map_shop = {}
    counter = 0

    for i in tqdm(range(len(shop_data))):
        n_ft = 0
        index_map_shop[counter] = (shop_data.loc[i, "title"], shop_data.loc[i, "image_links"])
        vector_ft = np.zeros(100)
        for word in shop_data.loc[i, "text"]:
            if word in modelFT.wv:
                vector_ft += modelFT.wv[word] * idfs.get(word, midf)
                n_ft += idfs.get(word, midf)
        if n_ft > 0:
            vector_ft = vector_ft / n_ft
        ft_index_shop.add_item(counter, vector_ft)
        counter += 1

    ft_index_shop.build(10)
    ft_index_shop.save('/content/drive/MyDrive/Colab Notebooks/telegram_bot/shop.ann')

    file_path_from = '/content/drive/MyDrive/Colab Notebooks/telegram_bot/index_shop.pkl'
    if not os.path.isfile(file_path_from):

        with open("/content/drive/MyDrive/Colab Notebooks/telegram_bot/index_shop.pkl", "wb") as f:
            pickle.dump(index_map_shop, f)


In [45]:
# Загрузим индексы

midf = np.mean(tfidf_vect.idf_)

ft_index_shop = annoy.AnnoyIndex(100, 'angular')
ft_index_shop.load('/content/drive/MyDrive/Colab Notebooks/telegram_bot/shop.ann')

index_map_shop = pd.read_pickle("/content/drive/MyDrive/Colab Notebooks/telegram_bot/index_shop.pkl")

In [46]:
# Основная функция преобразования текста в вектор х100

def embed_txt(txt, idfs, midf):
    n_ft = 0
    vector_ft = np.zeros(100)
    for word in txt:
        if word in modelFT.wv:
            vector_ft += modelFT.wv[word] * idfs.get(word, midf)
            n_ft += idfs.get(word, midf)
    return vector_ft / n_ft

In [47]:
# Пример получения индекса

ft_index_shop.get_nns_by_vector(np.ones(100)*20, 5, include_distances=True)

([73, 1030, 1072, 2112, 2577],
 [1.4264369010925293,
  1.4264369010925293,
  1.4264369010925293,
  1.4264369010925293,
  1.4264369010925293])

Создание своего бота в телеграмм:

@botfather

/start

/newbot - create a new bot

name1

name1_BOT

. ->. API

In [49]:

# заменить на свой токен
updater = Updater("6373890785:AAFiHVI9ps8STEtwH98fVj6jribCaw3Hygo", use_context=True) # Токен API к Telegram
dispatcher = updater.dispatcher

def startCommand(update, context):
    context.bot.send_message(chat_id=update.message.chat_id, text=f'Привет, давай пообщаемся?')

def textMessage(update, context):

    input_txt = preprocess_txt(update.message.text)
    vect = vectorizer.transform([" ".join(input_txt)])
    prediction = lr.predict(vect)

    # ПРОД
    if prediction[0] == 1:
        vect_ft = embed_txt(input_txt, idfs, midf)
        ft_index_shop_val = ft_index_shop.get_nns_by_vector(vect_ft, 5)
        for item in ft_index_shop_val:
            title, image = index_map_shop[item]
            context.bot.send_message(chat_id=update.message.chat_id, text=f"title: {title} image: {image}")
        return

    # QA
    vect_ft = embed_txt(input_txt, {}, 1)
    ft_index_val, distances = ft_index.get_nns_by_vector(vect_ft, 1, include_distances=True)

    #
    if distances[0] > 100.5:
        print(distances[0])
        context.bot.send_message(chat_id=update.message.chat_id, text=f"Моя твоя не понимать")
        return

    # Вопрос-Ответ
    context.bot.send_message(chat_id=update.message.chat_id, text=index_map[ft_index_val[0]])

start_command_handler = CommandHandler('start', startCommand)
text_message_handler = MessageHandler(Filters.text, textMessage)
dispatcher.add_handler(start_command_handler)
dispatcher.add_handler(text_message_handler)
updater.start_polling(clean=True)
updater.idle()